# NN analysis

### Imports

In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

import sys
from pathlib import Path
import importlib

# Sørg for at code/ er på path
sys.path.append(str(Path("..").resolve()))

from Implementations import prepare_data
importlib.reload(prepare_data)

from Implementations.prepare_data import prepare_data






### Data prep

In [5]:
import pandas as pd

df = pd.read_csv("../processed_player_value/nn_tabular_dataset.csv")

data = prepare_data(df, test_size=0.2, seed=42)
print(data["summary"])

X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["ylog_train"].ravel()
y_test  = data["ylog_test"].ravel()

feature_cols = data["feature_cols"]
groups_full = data["groups_full"]

X_train = data["X_train"]
X_test  = data["X_test"]

y_train = data["ylog_train"].ravel()
y_test  = data["ylog_test"].ravel()

X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train).unsqueeze(1)

X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test).unsqueeze(1)


{'n_rows_total': 278558, 'n_rows_used': 275230, 'n_features': 15, 'test_size': 0.2, 'seed': 42, 'numeric_only': True, 'drop_na': True, 'standardize': True, 'player_overlap_train_test': 0}


In [6]:
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=2048,
    shuffle=False
)


In [7]:
class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(d_in=X_train.shape[1])


In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
loss_fn = nn.MSELoss()

def train_epoch():
    model.train()
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

@torch.no_grad()
def eval_model():
    model.eval()
    preds = []
    for xb, _ in test_loader:
        preds.append(model(xb).numpy())
    yhat = np.vstack(preds).ravel()
    rmse = np.sqrt(mean_squared_error(y_test, yhat))
    r2 = r2_score(y_test, yhat)
    return rmse, r2

for epoch in range(1, 21):
    train_epoch()
    rmse, r2 = eval_model()
    print(f"Epoch {epoch:02d} | RMSE(log)={rmse:.4f} | R2(log)={r2:.4f}")


Epoch 01 | RMSE(log)=1.6034 | R2(log)=-0.1334
Epoch 02 | RMSE(log)=1.1331 | R2(log)=0.4340
Epoch 03 | RMSE(log)=1.0986 | R2(log)=0.4679
Epoch 04 | RMSE(log)=1.0898 | R2(log)=0.4764
Epoch 05 | RMSE(log)=1.0835 | R2(log)=0.4824
Epoch 06 | RMSE(log)=1.0838 | R2(log)=0.4821
Epoch 07 | RMSE(log)=1.0821 | R2(log)=0.4837
Epoch 08 | RMSE(log)=1.0746 | R2(log)=0.4908
Epoch 09 | RMSE(log)=1.0822 | R2(log)=0.4837
Epoch 10 | RMSE(log)=1.0755 | R2(log)=0.4900
Epoch 11 | RMSE(log)=1.0773 | R2(log)=0.4883
Epoch 12 | RMSE(log)=1.1123 | R2(log)=0.4545
Epoch 13 | RMSE(log)=1.0757 | R2(log)=0.4899
Epoch 14 | RMSE(log)=1.0860 | R2(log)=0.4801
Epoch 15 | RMSE(log)=1.0818 | R2(log)=0.4840
Epoch 16 | RMSE(log)=1.0762 | R2(log)=0.4893
Epoch 17 | RMSE(log)=1.0731 | R2(log)=0.4923
Epoch 18 | RMSE(log)=1.0853 | R2(log)=0.4807
Epoch 19 | RMSE(log)=1.0759 | R2(log)=0.4897
Epoch 20 | RMSE(log)=1.0806 | R2(log)=0.4851


### Testing different architectures

In [9]:
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, r2_score

# -----------------------------
# Assumes you already have:
# data = prepare_data_from_df(df, ...)
# X_train = data["X_train"], X_test = data["X_test"]
# y_train = data["ylog_train"].ravel(), y_test = data["ylog_test"].ravel()
# -----------------------------

X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["ylog_train"].ravel()
y_test  = data["ylog_test"].ravel()

# Torch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True, num_workers=0)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=2048, shuffle=False, num_workers=0)

device = "cpu"  # change to "cuda" only if stable
torch.manual_seed(42)
np.random.seed(42)


def make_activation(name: str):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError("activation must be 'relu' or 'tanh'")


class MLP(nn.Module):
    def __init__(self, d_in: int, n_layers: int, n_units: int, activation: str, dropout: float = 0.1):
        super().__init__()
        act = make_activation(activation)
        layers = []
        prev = d_in
        for _ in range(n_layers):
            layers.append(nn.Linear(prev, n_units))
            layers.append(act.__class__())  # new instance
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = n_units
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def eval_model(model: nn.Module):
    model.eval()
    preds = []
    trues = []
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = model(xb).cpu().numpy()
        preds.append(pred)
        trues.append(yb.numpy())
    yhat = np.vstack(preds).ravel()
    ytrue = np.vstack(trues).ravel()
    rmse = np.sqrt(mean_squared_error(ytrue, yhat))
    r2 = r2_score(ytrue, yhat)
    return rmse, r2


def train_one_config(
    n_layers: int,
    n_units: int,
    activation: str,
    *,
    lr: float = 5e-4,
    weight_decay: float = 1e-5,
    dropout: float = 0.1,
    max_epochs: int = 40,
    patience: int = 6,
    clip_norm: float = 5.0,
):
    # reproducible init per config (optional but nice)
    torch.manual_seed(42)
    model = MLP(d_in=X_train.shape[1], n_layers=n_layers, n_units=n_units, activation=activation, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_rmse = np.inf
    best_r2 = -np.inf
    best_epoch = 0
    best_state = None
    bad = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            opt.step()

        rmse, r2 = eval_model(model)

        if rmse < best_rmse - 1e-4:
            best_rmse = rmse
            best_r2 = r2
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state)

    # final metrics (best)
    rmse, r2 = eval_model(model)

    # count params
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        "activation": activation,
        "n_layers": n_layers,
        "n_units": n_units,
        "n_params": int(n_params),
        "best_epoch": int(best_epoch),
        "rmse_log": float(rmse),
        "r2_log": float(r2),
    }


def run_sweep(activation: str, layers_grid, units_grid):
    rows = []
    t0 = time.time()

    for L in layers_grid:
        for H in units_grid:
            res = train_one_config(L, H, activation)
            rows.append(res)
            print(f"[{activation}] layers={L}, units={H} -> RMSE(log)={res['rmse_log']:.4f}, R2(log)={res['r2_log']:.4f}, best_epoch={res['best_epoch']}")

    df_res = pd.DataFrame(rows).sort_values(["n_layers", "n_units"]).reset_index(drop=True)
    print(f"Done {activation} sweep in {time.time()-t0:.1f}s")
    return df_res


# -----------------------------
# Sweep grids (as proposed)
# -----------------------------
layers_grid = [1, 2, 3]
units_grid  = [32, 64, 128, 256]

relu_results = run_sweep("relu", layers_grid, units_grid)
tanh_results = run_sweep("tanh", layers_grid, units_grid)

all_results = pd.concat([relu_results, tanh_results], ignore_index=True)

display(all_results)

# Optional: save to CSV (recommended for reproducibility)
all_results.to_csv("mlp_arch_sweep_results.csv", index=False)
print("Saved: mlp_arch_sweep_results.csv")


[relu] layers=1, units=32 -> RMSE(log)=1.0721, R2(log)=0.4933, best_epoch=35
[relu] layers=1, units=64 -> RMSE(log)=1.0706, R2(log)=0.4946, best_epoch=38
[relu] layers=1, units=128 -> RMSE(log)=1.0691, R2(log)=0.4961, best_epoch=30
[relu] layers=1, units=256 -> RMSE(log)=1.0703, R2(log)=0.4949, best_epoch=12
[relu] layers=2, units=32 -> RMSE(log)=1.0653, R2(log)=0.4996, best_epoch=39
[relu] layers=2, units=64 -> RMSE(log)=1.0669, R2(log)=0.4982, best_epoch=16
[relu] layers=2, units=128 -> RMSE(log)=1.0683, R2(log)=0.4968, best_epoch=9
[relu] layers=2, units=256 -> RMSE(log)=1.0709, R2(log)=0.4944, best_epoch=7
[relu] layers=3, units=32 -> RMSE(log)=1.0686, R2(log)=0.4965, best_epoch=17
[relu] layers=3, units=64 -> RMSE(log)=1.0661, R2(log)=0.4989, best_epoch=25
[relu] layers=3, units=128 -> RMSE(log)=1.0673, R2(log)=0.4977, best_epoch=8
[relu] layers=3, units=256 -> RMSE(log)=1.0651, R2(log)=0.4998, best_epoch=13
Done relu sweep in 8983.4s
[tanh] layers=1, units=32 -> RMSE(log)=1.0821,

,activation,n_layers,n_units,n_params,best_epoch,rmse_log,r2_log
0,relu,1,32,545,35,1.072084,0.493258
1,relu,1,64,1089,38,1.070633,0.494628
2,relu,1,128,2177,30,1.069056,0.496116
3,relu,1,256,4353,12,1.070342,0.494903
4,relu,2,32,1601,39,1.065311,0.499640
5,relu,2,64,5249,16,1.066893,0.498153
6,relu,2,128,18689,9,1.068348,0.496783
7,relu,2,256,70145,7,1.070890,0.494385
8,relu,3,32,2657,17,1.068603,0.496543
9,relu,3,64,9409,25,1.066052,0.498944


Saved: mlp_arch_sweep_results.csv


In [10]:
# Create pivot tables for heatmaps
relu_r2 = relu_results.pivot(index="n_layers", columns="n_units", values="r2_log")
tanh_r2 = tanh_results.pivot(index="n_layers", columns="n_units", values="r2_log")

relu_rmse = relu_results.pivot(index="n_layers", columns="n_units", values="rmse_log")
tanh_rmse = tanh_results.pivot(index="n_layers", columns="n_units", values="rmse_log")

print("ReLU R2 matrix:\n", relu_r2)
print("\nTanh R2 matrix:\n", tanh_r2)


ReLU R2 matrix:
 n_units        32        64        128       256
n_layers                                        
1         0.493258  0.494628  0.496116  0.494903
2         0.499640  0.498153  0.496783  0.494385
3         0.496543  0.498944  0.497734  0.499803

Tanh R2 matrix:
 n_units        32        64        128       256
n_layers                                        
1         0.483790  0.488413  0.490609  0.490447
2         0.494924  0.496981  0.496530  0.497461
3         0.495730  0.497175  0.496012  0.495503
